<div dir="rtl" style="text-align:right">
<h1 style="text-align:right">دو جمع در یک بلوک واقعی</h1><p style="text-align:right"><b>پرسش آزمایش:</b> چرا ورودیِ جمع دوم، نتیجهٔ جمع اول است؟</p><p style="text-align:right">پیش‌نیاز: <a target="_self" href="http://127.0.0.1:8000/part-06/chapter-02/37-ffn.html"><bdi dir="ltr">37-ffn</bdi></a>، <a target="_self" href="http://127.0.0.1:8000/part-06/chapter-03/38-residual.html"><bdi dir="ltr">38-residual</bdi></a>، <a target="_self" href="http://127.0.0.1:8000/part-06/chapter-03/39-layernorm.html"><bdi dir="ltr">39-layernorm</bdi></a>، <a target="_self" href="http://127.0.0.1:8000/part-06/chapter-04/40-block.html"><bdi dir="ltr">40-block</bdi></a>، <a target="_self" href="http://127.0.0.1:8000/part-06/chapter-04/41-stack.html"><bdi dir="ltr">41-stack</bdi></a>، <a target="_self" href="http://127.0.0.1:8000/part-07/chapter-02/45-trace.html"><bdi dir="ltr">45-trace</bdi></a></p><p style="text-align:right">این دفتر مستقل است و به اجرای دفتر دیگری نیاز ندارد. از بالا به پایین اجرا کنید؛ برای اجرای دوباره از ابتدا، <bdi dir="ltr">Kernel</bdi> را <bdi dir="ltr">Restart</bdi> و سپس <bdi dir="ltr">Run All</bdi> کنید. برای بازکردن لینک درس‌ها، سرور کتاب باید روی پورت ۸۰۰۰ اجرا شده باشد؛ راهنمای نصب در <a href="../../docs/NOTEBOOKS.md"><bdi dir="ltr">docs/NOTEBOOKS.md</bdi></a> است.</p>
</div>

In [ ]:
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "mini_gpt" / "model.py").is_file()
             and (p / "data" / "sample.txt").is_file()), None)
if ROOT is None:
    raise RuntimeError("Keep notebooks inside the extracted project folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Python:", sys.executable)
print("Project:", ROOT)

import torch
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(17)

def inspect(name, value):
    print(name, "shape =", tuple(value.shape),
          "dtype =", value.dtype, "device =", value.device)


<div dir="rtl" style="text-align:right">
<p style="text-align:right">بلوک نخستِ <bdi dir="ltr">MiniGPT</bdi> واقعی را باز می‌کنیم. معماری این پروژه <bdi dir="ltr">Pre-Norm</bdi> است: <code dir="ltr" style="unicode-bidi:isolate;direction:ltr;text-align:left">y=x+Attention(LN₁(x))</code> و سپس <code dir="ltr" style="unicode-bidi:isolate;direction:ltr;text-align:left">out=y+FFN(LN₂(y))</code>. ترتیب را با <bdi dir="ltr">Post-Norm</bdi> جابه‌جا نکنید. وزن‌ها تصادفی و ثابت‌اند.</p>
</div>

In [ ]:
from mini_gpt.config import ModelConfig
from mini_gpt.model import MiniGPT
config = ModelConfig(vocab_size=12,context_length=8,embedding_dim=12,
                     num_heads=3,num_layers=2,dropout=0.)
model = MiniGPT(config).eval()
ids = torch.tensor([[1,2,3,4,5]],dtype=torch.long)
targets = torch.tensor([[2,3,4,5,6]],dtype=torch.long)
trace = {}
with torch.no_grad():
    logits, loss = model(ids,targets,trace=trace)
    block = model.blocks[0]
    x = trace["combined_embedding"]
    normalized_1 = block.norm_1(x)
    update_1 = block.attention(normalized_1)
    y = x + update_1
    normalized_2 = block.norm_2(y)
    update_2 = block.feed_forward(normalized_2)
    out = y + update_2
    torch.testing.assert_close(update_2,trace["layers"][0]["feed_forward"])
    torch.testing.assert_close(out,trace["layers"][0]["output"])
    hidden = out
    for later in model.blocks[1:]:
        hidden = later(hidden)
    reconstructed = model.language_model_head(model.final_norm(hidden))
    torch.testing.assert_close(reconstructed,logits)
for name,value in [("IDs",ids),("Token Embedding",trace["token_embedding"]),
                   ("Position Embedding",trace["position_embedding"]),("block input",x),
                   ("LN1",normalized_1),("Attention update",update_1),
                   ("first sum",y),("LN2",normalized_2),("FFN update",update_2),
                   ("block output",out),("logits",logits)]:
    inspect(name,value)
print("Loss:",loss.item())


<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">حذف میان‌بر را دقیق تعریف کنیم</h2><p style="text-align:right">گزینهٔ <bdi dir="ltr">residual</bdi>=<bdi dir="ltr">False</bdi> هر دو جمع مستقیم را حذف می‌کند؛ <bdi dir="ltr">Layer Normalization</bdi> و <bdi dir="ltr">Mask</bdi> باقی می‌مانند. این کار فقط کم‌کردن <bdi dir="ltr">x</bdi> از خروجی نهایی نیست، چون ورودی <bdi dir="ltr">FFN</bdi> هم عوض می‌شود. قبل از اجرا <bdi dir="ltr">Shape</bdi> و علّیت را پیش‌بینی کنید.</p>
</div>

In [ ]:
with torch.no_grad():
    removed = block(x,residual=False)
    expected = block.feed_forward(block.norm_2(block.attention(block.norm_1(x))))
    torch.testing.assert_close(removed,expected)
    assert removed.shape == out.shape
    changed = ids.clone()
    changed[:,-2:] = torch.tensor([9,10])
    for residual in (True,False):
        original,_ = model(ids,residual=residual)
        modified,_ = model(changed,residual=residual)
        torch.testing.assert_close(original[:,:3],modified[:,:3],rtol=0,atol=1e-7)
print("Shape and causal-prefix checks passed.")
try:
    model(torch.ones(1,config.context_length+1,dtype=torch.long))
except ValueError as error:
    print("Expected context limit:",error)
else:
    raise AssertionError("Expected context-length guard")


<div dir="rtl" style="text-align:right">
<p style="text-align:right"><b>تمرین:</b> یک مدل تازه با <bdi dir="ltr">context_length</bdi>=16 بسازید و تعداد <bdi dir="ltr">Parameter</bdi>های جدول موقعیت را مقایسه کنید. سپس فقط <bdi dir="ltr">C</bdi> را، با رعایت تقسیم‌پذیری بر <bdi dir="ltr">H</bdi>، تغییر دهید. <bdi dir="ltr">Shape</bdi>ها و شمار <bdi dir="ltr">Parameter</bdi>ها را پیش‌بینی کنید؛ انتظار یکسان‌ماندن خروجی عددی مدل تازه نداریم. این آزمون بدون آموزش دربارهٔ کیفیت مدل با یا بدون <bdi dir="ltr">Residual</bdi> داوری نمی‌کند.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">برگشت به کتاب</h2><p style="text-align:right">پیش‌بینی و نتیجهٔ اجرا را کنار هم بنویسید؛ اگر تفاوتی داشتند، دلیلش را توضیح دهید. سپس به <a target="_self" href="http://127.0.0.1:8000/part-07/chapter-02/45-trace.html">درس مرتبط</a> برگردید و نتیجه را با توضیح آن مقایسه کنید.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">تمرین تکمیلی: همان وزن‌ها، ترتیب متفاوتِ <bdi dir="ltr">Normalization</bdi></h2>
<p style="text-align:right">تفاوت <bdi dir="ltr">Pre-Norm</bdi> و یک مسیر <bdi dir="ltr">Post-Norm</bdi> را با کنترل وزن‌ها بررسی کنید. پیش‌نیاز: دو جمع و محل <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">norm_1/norm_2</code> در بلوک واقعی این دفتر را دنبال کرده باشید. مثال‌های قبلی این دفتر را نگه داشته‌ایم. اکنون دو تابع <bdi dir="ltr">TODO</bdi> را خودتان بنویسید؛ <bdi dir="ltr">INCOMPLETE</bdi> یعنی کار هنوز تمام نشده است.</p>
</div>

In [ ]:
from pathlib import Path
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">قبل از اجرا، پیش‌بینی کنید</h2>
<p style="text-align:right">اگر <bdi dir="ltr">Layer Normalization</bdi> را از ورودی زیرلایه به بعد از جمع منتقل کنیم، <bdi dir="ltr">Shape</bdi> تغییر می‌کند؟ آیا همان عددها را انتظار داریم؟</p>
</div>

<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی من: …</p></div>

In [ ]:
import math
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
from copy import deepcopy
from mini_gpt.config import ModelConfig
from mini_gpt.transformer import TransformerBlock
config = ModelConfig(12,8,8,2,1,0.)
block = TransformerBlock(config).eval()
x = torch.randn(2,4,8)
print('actual project architecture: Pre-Norm')

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">این بار شما کد بنویسید</h2>
<p style="text-align:right">تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">postnorm_candidate(block,x)</code> این مسیر مقایسه‌ای را با همان زیرلایه‌ها بسازد: <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">y=norm_1(x+attention(x))</code> و <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">out=norm_2(y+feed_forward(y))</code>. این تابع معماری پروژه نیست؛ هدف دیدن اثر ترتیب است، نه اصلاح <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">TransformerBlock</code>.</p>
</div>

In [ ]:
def postnorm_candidate(block, x):
    # TODO
    return None

In [ ]:
def test_exercise():
    result = postnorm_candidate(block,x)
    if result is None: return False
    y = block.norm_1(x+block.attention(x))
    torch.testing.assert_close(result,block.norm_2(y+block.feed_forward(y)))
    assert result.shape == x.shape
    zero = deepcopy(block)
    with torch.no_grad():
        for sublayer in (zero.attention,zero.feed_forward):
            for parameter in sublayer.parameters(): parameter.zero_()
    torch.testing.assert_close(postnorm_candidate(zero,x),zero.norm_2(zero.norm_1(x)))
    z = torch.randn(1,1,8)
    a = block.norm_1(z+block.attention(z))
    torch.testing.assert_close(postnorm_candidate(block,z),block.norm_2(a+block.feed_forward(a)))
    return True

exercise_complete = test_exercise()
print("PASS" if exercise_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">فقط یک عامل را تغییر دهید</h2>
<p style="text-align:right">فقط ترتیب <bdi dir="ltr">Normalization</bdi> را عوض کنید؛ هیچ <bdi dir="ltr">Layer</bdi> یا وزن تازه‌ای نسازید. اختلاف این اجرا دربارهٔ کیفیت آموزش دو معماری داوری نمی‌کند.</p>
</div>

In [ ]:
with torch.no_grad():
    pre = block(x)
    y = block.norm_1(x+block.attention(x))
    post = block.norm_2(y+block.feed_forward(y))
print('same shape:',pre.shape == post.shape,'ordering difference:',(pre-post).abs().mean().item())

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">خرابی را پیدا کنید</h2>
<p style="text-align:right">مقایسهٔ دو بلوکِ تازه، اثر ترتیب را با تفاوت وزن‌ها مخلوط می‌کند. تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">align_weights(reference,candidate)</code> مقدار <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">state_dict</code> شیء <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">reference</code> را در <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">candidate</code> بارگذاری و <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">candidate</code> را برگرداند؛ دو شیء همچنان مستقل بمانند.</p>
</div>

In [ ]:
candidate = TransformerBlock(config).eval()
print('uncontrolled weight difference:',(candidate.attention.qkv.weight-block.attention.qkv.weight).abs().max().item())

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">اصلاح را خودتان بنویسید</h2>
<p style="text-align:right">علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def align_weights(reference, candidate):
    # TODO
    return None

In [ ]:
def test_repair():
    result = align_weights(block,candidate)
    if result is None: return False
    assert result is candidate and result is not block
    for name,value in block.state_dict().items():
        assert torch.equal(value,candidate.state_dict()[name])
    assert candidate.attention.qkv.weight is not block.attention.qkv.weight
    torch.testing.assert_close(candidate(x),block(x))
    return True

repair_complete = test_repair()
print("PASS" if repair_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">در <bdi dir="ltr">Mini-GPT</bdi> کجا به کار می‌آید؟</h2>
<p style="text-align:right"><code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">TransformerBlock</code> پروژه <bdi dir="ltr">Pre-Norm</bdi> باقی می‌ماند. آزمایش مقایسه‌ای از همان قطعه‌ها استفاده می‌کند تا نام معماری به محل دقیق عمل‌ها وصل شود؛ پیاده‌سازی اصلی پروژه تغییری نمی‌کند.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">با زبان خودتان توضیح دهید</h2>
<p style="text-align:right">برای نسبت‌دادن اختلاف خروجی به ترتیب <bdi dir="ltr">Layer</bdi>‌ها، چه عامل‌هایی را در این مقایسه ثابت نگه داشتید؟</p>
</div>
<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی و مشاهدهٔ من: …</p><p style="text-align:right">علت خرابی و اصلاح من: …</p></div>

<div dir="rtl" style="text-align:right"><p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-07/chapter-02/45-trace.html">بازگشت به درس مرتبط</a> · <a target="_self" href="http://127.0.0.1:8000/answers/lab-10_block_trace.html">فقط پس از تلاش: پاسخ مرجع تمرین تکمیلی</a></p></div>